In [2]:
import time

running_time = time.time()

In [3]:
import sys
sys.path.append('../')

import logging
logging.getLogger('matplotlib').setLevel(logging.CRITICAL)
logging.getLogger('graphein').setLevel(logging.INFO)

In [4]:
import pandas as pd
import numpy as np

from tqdm import tqdm
import networkx as nx

import warnings
warnings.filterwarnings("ignore")


In [5]:
from pkg.PDBGraphStore import PDBGraphStore as store
from pkg.MemoryMeasuring import MemoryMeasuring as m

In [6]:
from graphein.protein.config import ProteinGraphConfig
from graphein.protein.edges.distance import (
    add_hydrogen_bond_interactions, 
    add_peptide_bonds, 
    add_k_nn_edges, 
    add_aromatic_interactions,
    add_aromatic_sulphur_interactions
)
from graphein.protein.graphs import construct_graph
from graphein.ml.conversion import GraphFormatConvertor

from graphein.protein.utils import download_pdb
import os

from functools import partial

In [7]:
def fit(graph: nx.Graph) -> nx.Graph:
    g_config = graph.graph["config"]
    g_pdb_code = graph.graph["pdb_code"]
    graph.graph.clear()
    graph.graph['config'] = g_config
    graph.graph['pdb_code'] = g_pdb_code

    return graph

In [8]:
constructors = {
    # "edge_construction_functions": [partial(add_k_nn_edges, k=3, long_interaction_threshold=0)],
    "graph_metadata_functions": [fit],
    "edge_construction_functions": [add_hydrogen_bond_interactions, add_aromatic_sulphur_interactions, add_aromatic_interactions, add_peptide_bonds],
    'granularity': 'CA',
    #"node_metadata_functions": [add_dssp_feature]
}

config = ProteinGraphConfig(**constructors)
print(config.dict())

{'granularity': 'CA', 'keep_hets': [], 'insertions': True, 'alt_locs': 'max_occupancy', 'pdb_dir': None, 'verbose': False, 'exclude_waters': True, 'deprotonate': False, 'protein_df_processing_functions': None, 'edge_construction_functions': [<function add_hydrogen_bond_interactions at 0x7efca3508400>, <function add_aromatic_sulphur_interactions at 0x7efca35085e0>, <function add_aromatic_interactions at 0x7efca3508540>, <function add_peptide_bonds at 0x7efca3508220>], 'node_metadata_functions': [<function meiler_embedding at 0x7efca350b100>], 'edge_metadata_functions': None, 'graph_metadata_functions': [<function fit at 0x7efbccdb8ae0>], 'get_contacts_config': None, 'dssp_config': None}


In [9]:
df = pd.read_csv(
    '../../data/scope.2.08.txt',
    sep="\t",
    comment="#",
    header=None,
)

df.columns = [
    "domain_id",
    "pdb",
    "region",
    "sccs",
    "sunid",
    "hierarchy"
]

print(len(df))

df.head(3)
df = df.sample(n=3000, random_state=42)

df["superfamily"] = df["sccs"].str.extract(r"^([a-z]\.\d+\.\d+)")

print(df.head(3))
print(len(df))

344851
       domain_id   pdb   region      sccs   sunid  \
168673   d1gytd1  1gyt  D:1-178  c.50.1.1   70771   
328794   d3quca2  3quc    A:0-0   l.1.1.1  294841   
150848   d6b7fa1  6b7f  A:2-159  c.26.1.3  349442   

                                                hierarchy superfamily  
168673  cl=51349,cf=52948,sf=52949,fa=52950,dm=52951,s...      c.50.1  
328794  cl=310555,cf=310573,sf=310607,fa=310682,dm=310...       l.1.1  
150848  cl=51349,cf=52373,sf=52374,fa=52397,dm=52398,s...      c.26.1  
3000


In [10]:
pdb_store = store()

In [11]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

In [12]:
df["superfamily_id"] = encoder.fit_transform(df["superfamily"])

In [13]:
print(df['superfamily_id'].head())
print(len(df))

168673    319
328794    655
150848    300
24057      87
240283    547
Name: superfamily_id, dtype: int64
3000


In [14]:
pdb_data_path = "../../data/pdb_files/"

for idx, pdb_code in enumerate(tqdm(df['pdb'])):
    if os.path.exists(f'{pdb_data_path}/{pdb_code}.pdb'):
        pdb_file = os.path.abspath(f'{pdb_data_path}/{pdb_code}.pdb')
    else:
        try:
            pdb_file = download_pdb(pdb_code, f'{pdb_data_path}/')
        except Exception as e:
            print(str(idx) + "processing error....")
            pass

    if pdb_file == None:
        print(str(idx) + "processing error....")
        pass
    
    try:
        graph = construct_graph(config=config, path=pdb_file, pdb_code=pdb_code)
        pdb_store.insert({pdb_code: graph})

    except Exception as e:
        print(str(idx) + ' processing error...')
        print(e)
        pass

graph = None

  0%|          | 0/3000 [00:00<?, ?it/s]

Output()

  0%|          | 1/3000 [00:01<1:39:24,  1.99s/it]

Output()

Output()

...

100%|█████████▉| 2998/3000 [20:14<00:00,  5.84it/s]

Output()

100%|█████████▉| 2999/3000 [20:14<00:00,  5.17it/s]

Output()

100%|██████████| 3000/3000 [20:14<00:00,  2.47it/s]


In [15]:
# from pkg.MemoryMeasuring import MemoryMeasuring as m

# memo = m(pdb_store)

In [16]:
# print(memo.total_memory())

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import (
    SAGEConv,
    BatchNorm,
    global_mean_pool
)


class ProteinGraphClassifier(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_classes,
        dropout=0.3
    ):
        super().__init__()

        self.conv1 = SAGEConv(input_dim, hidden_dim)
        self.bn1 = BatchNorm(hidden_dim)

        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.bn2 = BatchNorm(hidden_dim)

        self.conv3 = SAGEConv(hidden_dim, hidden_dim)
        self.bn3 = BatchNorm(hidden_dim)

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, data):

        x = torch.cat(
            [
                data.meiler.float(),
                data.coords.float()
            ],
            dim=1
        )
        edge_index = data.edge_index
        batch = data.batch

        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)

        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)

        x = global_mean_pool(x, batch)

        out = self.classifier(x)

        return out

In [18]:
from sklearn.model_selection import train_test_split

In [19]:
counts = df["superfamily_id"].value_counts()
print(len(df))
valid_families = counts[counts >= 6].index

df = df[df["superfamily_id"].isin(valid_families)]

print(len(df))

3000
2033


In [20]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["superfamily_id"],
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["superfamily_id"],
    random_state=42
)

temp_df = None

In [21]:
print(len(train_df))
print(len(valid_df))
print(len(test_df))

1423
305
305


In [22]:
print(train_df.head(1))

       domain_id   pdb region     sccs  sunid  \
306554   d1jzym_  1jzy     M:  i.1.1.2  67891   

                                                hierarchy superfamily  \
306554  cl=58117,cf=58118,sf=58119,fa=58124,dm=58125,s...       i.1.1   

        superfamily_id  
306554             638  


In [23]:
convertor = GraphFormatConvertor(src_format="nx", dst_format="pyg", columns=["coords", "edge_index", "meiler"])

In [24]:
import traceback

In [25]:
train_ds = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
    pdb_code = row["pdb"]
    sf_id = row["superfamily_id"]
    try:
        g = pdb_store.extract(pdb_code)
        aux = {}

        aux[pdb_code] = convertor(g)
        aux[pdb_code].y = torch.tensor([sf_id], dtype=torch.long)
        train_ds.append(aux[pdb_code])

    except Exception:
        print(f'Erro em {pdb_code}')
        traceback.print_exc()

100%|██████████| 1423/1423 [06:01<00:00,  3.93it/s]


In [26]:
valid_ds = []

for _, row in tqdm(valid_df.iterrows(), total=len(valid_df)):
    pdb_code = row["pdb"]
    sf_id = row["superfamily_id"]
    try:
        g = pdb_store.extract(pdb_code)
        aux = {}

        aux[pdb_code] = convertor(g)
        aux[pdb_code].y = torch.tensor([sf_id], dtype=torch.long)
        valid_ds.append(aux[pdb_code])

    except Exception:
        print(f'Erro em {pdb_code}')
        traceback.print_exc()

 13%|█▎        | 39/305 [00:04<00:29,  8.97it/s]Traceback (most recent call last):
  File "/tmp/ipykernel_165/3958880290.py", line 7, in <module>
    g = pdb_store.extract(pdb_code)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/app/src/PDBGraphStore_notebooks/../pkg/PDBGraphStore.py", line 369, in extract
    return extract()
           ^^^^^^^^^
  File "/home/jovyan/app/src/PDBGraphStore_notebooks/../pkg/PDBGraphStore.py", line 352, in extract
    pdb_id = self.__body_parts["pdb_code_to_id"][pdb_upper]
             ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^
KeyError: '4GXV'


Erro em 4gxv


100%|██████████| 305/305 [01:14<00:00,  4.09it/s]


In [27]:
test_ds = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    pdb_code = row["pdb"]
    sf_id = row["superfamily_id"]
    try:
        g = pdb_store.extract(pdb_code)
        aux = {}

        aux[pdb_code] = convertor(g)
        aux[pdb_code].y = torch.tensor([sf_id], dtype=torch.long)
        test_ds.append(aux[pdb_code])

    except Exception:
        print(f'Erro em {pdb_code}')
        traceback.print_exc()

100%|██████████| 305/305 [01:09<00:00,  4.40it/s]


In [28]:
from torch_geometric.loader import DataLoader

# Create dataloaders
train_loader = DataLoader(train_ds, batch_size=16, shuffle=False, drop_last=True)
valid_loader = DataLoader(valid_ds, batch_size=16, shuffle=False, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=16, drop_last=True)

In [29]:
num_node_features = 10
num_classes = len(encoder.classes_)
print(num_classes)

model = ProteinGraphClassifier(
    input_dim=num_node_features,
    hidden_dim=128,
    num_classes=num_classes
)

656


In [30]:
criterion = nn.CrossEntropyLoss()

In [31]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

In [32]:
def train(model, loader, optimizer, criterion, device):

    model.train()

    total_loss = 0

    for batch in loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        out = model(batch)

        loss = criterion(out, batch.y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [33]:
@torch.no_grad()
def evaluate(model, loader, device):

    model.eval()

    correct = 0
    total = 0

    for batch in loader:

        batch = batch.to(device)

        out = model(batch)

        pred = out.argmax(dim=1)

        correct += (pred == batch.y).sum().item()
        total += batch.y.size(0)

    return correct / total

In [34]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

for epoch in range(1, 101):

    loss = train(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    train_acc = evaluate(
        model,
        train_loader,
        device
    )

    valid_acc = evaluate(
        model,
        valid_loader,
        device
    )

    print(
        f"Epoch {epoch:03d} | "
        f"Loss {loss:.4f} | "
        f"Train {train_acc:.4f} | "
        f"Valid {valid_acc:.4f}"
    )

Epoch 001 | Loss 5.0487 | Train 0.1754 | Valid 0.1776
Epoch 002 | Loss 4.1678 | Train 0.1768 | Valid 0.1743
Epoch 003 | Loss 4.0397 | Train 0.1797 | Valid 0.1809
Epoch 004 | Loss 3.9929 | Train 0.1776 | Valid 0.1809
Epoch 005 | Loss 3.9255 | Train 0.1783 | Valid 0.1809
Epoch 006 | Loss 3.8852 | Train 0.1832 | Valid 0.1809
Epoch 007 | Loss 3.8180 | Train 0.1868 | Valid 0.1809
Epoch 008 | Loss 3.7678 | Train 0.1903 | Valid 0.1842
Epoch 009 | Loss 3.7280 | Train 0.1882 | Valid 0.1809
Epoch 010 | Loss 3.6808 | Train 0.1861 | Valid 0.1743
Epoch 011 | Loss 3.6303 | Train 0.1875 | Valid 0.1776
Epoch 012 | Loss 3.6007 | Train 0.1882 | Valid 0.1842
Epoch 013 | Loss 3.5611 | Train 0.1903 | Valid 0.1809
Epoch 014 | Loss 3.5205 | Train 0.1882 | Valid 0.1809
Epoch 015 | Loss 3.5136 | Train 0.1946 | Valid 0.1842
Epoch 016 | Loss 3.4527 | Train 0.1903 | Valid 0.1809
Epoch 017 | Loss 3.4031 | Train 0.1932 | Valid 0.1809
Epoch 018 | Loss 3.3501 | Train 0.1946 | Valid 0.1809
Epoch 019 | Loss 3.3158 | Tr

In [35]:
with open('gnn_time.txt', 'a') as f:
    f.write(f'\n{str(time.time() - running_time)}')